# 后训练演进：从 Chatbot 到 Agent


> 前几讲各处理一类方法：06 讲给出强化学习的算法引擎（STaR、GRPO、DAPO），07 讲展示 Agent 自我改进的形态，08 讲把搜索与深度研究做成一个具体的 Agent。它们分散在不同章节，各自回答一个局部问题。
>
> 本讲把它们收拢成一条时间线，回答一个总问题：预训练只给了模型**续写文本**的能力，它怎样变成听话的助手，又怎样变成会用工具的 Agent。我们跟着一个模型走完它的一生，看每一阶段喂进去什么新**信号**，以及训练信号怎么从人类手里一步步交到环境手里。

我们问一个只有预训练能力的模型："12 加 8 等于几？"

它不回答 20。而是顺着这句话往下续写——可能续出"等于几呢"，也可能续出"我们一起算一算"。

原因在于：预训练教它的只是把一句话接下去，它根本不知道这是一道待回答的题。

要让模型从"会接话"变成"会干活"，人类在预训练之后又做了大量训练：先拿人类写好的问答示例教它照着做，再让标注员给它的回答排序，后来干脆用可以验证的答案、甚至环境执行的结果来训练它。预训练给的是**续写文本**的能力，这些发生在预训练之后、让模型更好用的训练，统称**后训练**。

这一讲讲的就是这条路。四段训练各有名字——**SFT**、**RLHF**、**RLVR**、Agent 后训练——我们看每一段喂给模型什么信号。学完这一讲，能说清这几种信号各自解决什么问题，也能看懂 SFT、RLHF、RLVR 这些名词指什么。第一节先看起点：一个只有预训练能力的模型，面对指令会输出什么。

## 1. 后训练是什么：从续写器到听话的模型

这一节解决一个问题：我们前面反复说"预训练只教会模型续写文本"，那"不会听指令"到底长什么样。先看清预训练模型的本来面目，它是一个文本续写器。

一个只有预训练能力的模型，只做一件事：给定任何开头，输出最像"互联网文本的继续"的下一个字符。把一段指令拼在开头，它也会顺着把指令续写下去，而不是把指令当作待执行的任务。预训练模型不缺少能力，缺少的是方向——它优化的目标不是用户意图。

从续写器到能完成任务的 Agent，训练信号在四个阶段里换了四种来源。下表是整讲的地图，每一行回答两个问题：模型从哪种信号里学到什么，以及这种信号为什么在下个阶段被替换。

| 阶段 | 信号来源 | 数据形态 | 目标 | 代表工作 |
|---|---|---|---|---|
| SFT | 人类示范 | (指令, 期望回复) | 学怎么做 | FLAN、InstructGPT-SFT |
| RLHF | 人类偏好 | 候选回复排序 | 学什么是好 | InstructGPT、ChatGPT |
| RLVR | 可验证正确性 | 规则判据 | 学什么是正确 | DeepSeek-R1、DAPO |
| Agent 后训练 | 环境执行结果 | 整条轨迹 + 成败 | 学什么动作序列能完成任务 | RLEF、WebRL、MiRA |

先用一个最小的续写器建立"模型不会听话"的直觉。

地图表格里每一行，都对应一类"喂给模型的数据"。信号来源不同，单个样本的样子就不同。把四个阶段各写一条样本，差别就落在纸面上：

| 阶段 | 一个样本的样子 | 模型从这条样本里学到的信号 |
|:---|:---|:---|
| SFT | 指令："把这句话翻译成英文：你好" + 用户写好的译文 | 示范里写了什么，照着写 |
| RLHF | 同一指令的 4 个候选回复 + 排序：回复 2 最好、回复 4 次之 | 哪个回复更讨人喜欢 |
| RLVR | 题目"37 + 48" + 标准答案 85；判据：答案是否等于 85 | 客观对不对 |
| Agent 后训练 | 一条轨迹：调用搜索 → 读返回 → 执行代码 → 返回结果，末尾一个成功或失败 | 这串动作能不能完成任务 |

看最后一列。SFT 给的是"应该写什么"，RLHF 给的是"哪一个更好"，RLVR 给的是"答案对不对"，Agent 后训练给的是"这串动作行不行"。信号从"人类写的"一路变成"环境判的"。整讲主线里说的迁移，迁移的就是这列信号：模型去优化的目标，从模仿人类文本，变成偏好，再变成可验证的正确性，最后变成环境执行结果。

下面回到出发点，看一个只会续写的模型面对指令会输出什么。



In [ ]:
import numpy as np
from collections import defaultdict

# 迷你语料：只有正常对话，没有"指令-执行"这种格式
corpus = ["天气不错我们出门散步", "晚餐吃什么我想吃面条", "今天的工作完成了早点休息"]

# 字符级 bigram 统计：记录每个字符后各字符出现的次数
cnt = defaultdict(lambda: defaultdict(int))
alphabet = set()
for sent in corpus:
    for ch in sent:
        alphabet.add(ch)
    for ch, nxt in zip(sent, sent[1:]):
        cnt[ch][nxt] += 1
alphabet = sorted(alphabet)


def next_char(ch):
    """按条件概率从 ch 的下一个字符分布里采样。"""
    if ch not in cnt or len(cnt[ch]) == 0:
        return np.random.choice(alphabet)
    options = list(cnt[ch].keys())
    probs = np.array([cnt[ch][c] for c in options], dtype=float)
    probs /= probs.sum()
    return np.random.choice(options, p=probs)


def continue_text(prefix, length=24):
    """从 prefix 出发续写 length 个字符。"""
    out = list(prefix)
    ch = out[-1]
    for _ in range(length):
        nxt = next_char(ch)
        out.append(nxt)
        ch = nxt
    return "".join(out)


np.random.seed(42)
prompt = "用户：请计算 12 加 8 等于几？"
print("输入指令：", prompt)
print("模型续写：", continue_text(prompt))
print()
print("关键观察：续写器没有回答数字，而是顺着指令写了一段像对话的文本。")


上面这格演示的续写过程可以拆开看。prompt 最后一个字符是"？"，而语料里"？"从未作为任何字符的下一个出现，所以 next_char 走不进计数表，只能从全部字符里均匀随机挑一个。从那个字符开始，续写才沿语料里的相邻统计走。

于是输出是"休息""想吃面条""早点休息""作完成了"这类片段的拼接，它们都是三个句子里的相邻字符组合。句子末尾的字符（步、条、息）在语料里没有后继，遇到它们也会回到均匀乱选，所以输出频繁跳变。

模型没有回答"20"。因为"指令-执行"这种模式在语料里从未出现：三句话全是陈述句，没有一句教会它"用户提问之后应该输出答案"。它不是不会算 12+8，而是它优化的目标里根本没有"回答问题"这件事。

这就是续写器与助手的差别。续写器优化的是"接下来最像语料的字符"，助手需要优化"用户要什么"。后者靠的就是第一节地图里那四种信号。



## 2. SFT 与 RLHF：Chatbot 时代的对齐

上一节我们看到，续写器面对指令只会往下写，不会回答。这一节解决这个问题：我们怎么让它学会回答。答案是把"该怎么回答"教给它，让模型的行为与人类期望一致，这个过程叫对齐。教法有两种，这一节各展开一种。

第一种教法，把人类写好的问答示例直接喂给模型，让它照着模仿。这种训练叫 `SFT`（监督微调）。做法是收集一批 (指令, 期望回复)，用交叉熵微调模型，让示范里的输出概率变高。SFT 只做这一件事——它不引入任何"好坏"的判断，示范里没有出现的回复，模型不会学到，也无法超越示教者的水平。

第二种教法，先让模型对同一指令输出多个候选，请标注员给这些候选排序，再让模型学着把分高的回复选出来。这种训练叫 `RLHF`（基于人类反馈的强化学习）。具体做法是先训练一个`RM`（奖励模型）给回复打分，再用强化学习最大化 RM 分数，同时用 KL 项约束策略不要偏离 SFT 模型太远。RM 分数是标注偏好的标量代理，只能衡量"这段文本好不好"。

下面用一套 toy 数据，把 SFT 的交叉熵和 RLHF 的优化目标逐项手算，看同一组概率被推向哪里。RLVR 与 Agent 后训练的损失形态留在第三节，届时一并对比。

后面代码里的数字要想真正看懂，先把两个基本概念对齐。

第一个是损失函数。训练模型就是在调参数 θ，让某个损失函数 L(θ) 变小。L 是"模型当前表现好坏"的一个数字：表现越接近我们希望，L 越小。梯度下降就是反复朝"让 L 变小的方向"调整 θ。所谓不同的训练目标，区别就在 L 长什么样、L 里的信号从哪来。

第二个是 logits 与 softmax。模型不直接输出概率，而是对词表里每个 token 输出一个未归一化的分数 logits，再经 softmax 变成概率。softmax 会把 logits 最大的 token 的概率抬到最高，但所有 token 的概率之和恒为 1。概率大，表示模型更倾向输出这个 token。

下面用只有 4 个 token（A、B、C、D）的玩具设定，把三种目标各算一遍。初始 logits 全为 0，经 softmax 后每个 token 的概率都是 0.25。三份信号分别是：示范 token 是 B；RM 分数 [0.5, 1.0, -0.2, 0.0]；正确性 [0, 1, 0, 1]（B、D 正确）。同一个初始模型、同一份数据，三种目标将给出三组不同的梯度。



In [ ]:
import numpy as np


def softmax(x):
    """把 logits 沿最后一维归一化成概率分布。"""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


# toy 设定：一条指令 x，模型的输出词表只有 4 个 token
tokens = ["A", "B", "C", "D"]
logits = np.zeros(4)                          # 初始 logits 全 0 → 均匀分布
probs = softmax(logits)

# 三类标签，对应三种时代的监督信号
demo_token = "B"                              # SFT：人类示范期望回复是 B
rm_scores = np.array([0.5, 1.0, -0.2, 0.0])   # RLHF：RM 给四个候选打分
correctness = np.array([0.0, 1.0, 0.0, 1.0])  # RLVR：B 与 D 正确

print("token        :", tokens)
print("当前概率 π   :", np.round(probs, 3))
print("SFT 示范 token:", demo_token)
print("RLHF RM 分数 :", rm_scores)
print("RLVR 正确性  :", correctness)


SFT 的损失就是输出序列的交叉熵。在单 token 的设定里，它退化为

$$L_{\mathrm{SFT}} = -\log \pi_\theta(y_{\mathrm{demo}}).$$

梯度只把示范 token 的概率推高。下面扫描示范 token 的概率从 0.1 到 0.9，观察损失怎样随概率变化，再用数值差分看梯度方向。


In [ ]:
import numpy as np

demo_idx = 1  # token B

# 扫描示范 token 的概率，看 SFT 损失随概率的变化
print("示范概率 p(B) | SFT 损失 -log p")
for p in np.arange(0.1, 0.95, 0.1):
    print(f"      {p:.1f}      |   {-np.log(p):.4f}")
print()


def sft_loss(theta):
    """单 token 设定下的 SFT 交叉熵损失。"""
    return -np.log(softmax(theta)[demo_idx])


theta = np.zeros(4)
eps = 1e-4
grad = np.array([(sft_loss(theta + eps * np.eye(4)[j]) -
                  sft_loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                 for j in range(4)])
print("SFT 梯度 dL/dθ :", np.round(grad, 3))
print("关键观察：示范 token B 的梯度为负（概率被推高），其余 token 梯度为正（概率被压低）。")


把 SFT 损失的手算过程拆开，让每个数字都有来源。

交叉熵的直观含义：-log p 在 p 接近 1 时趋于 0，在 p 接近 0 时趋于正无穷。示范 token 的概率越接近 1，损失越小。把 p(B)=0.25 代入：0.25 = 1/4，log(1/4) = -1.386，所以 L = -log 0.25 = 1.386。

梯度可以解析求。对 L = -log p(B) 求导，利用 softmax 的导数，得到

$$\frac{\partial L}{\partial \theta_j} = p_j - \mathbb{1}\{j = B\}.$$

初始 p 全是 0.25，于是梯度是 [0.25, -0.75, 0.25, 0.25]。示范 token B 的梯度为负，其余为正。梯度下降按 θ ← θ - lr·dL/dθ 更新，用学习率 1 走一步：

| token | 初始 p_j | 梯度 | 更新后 θ | 更新后 p_j |
|:---|:---|:---|:---|:---|
| A | 0.25 | +0.25 | -0.25 | 0.175 |
| B | 0.25 | -0.75 | +0.75 | 0.475 |
| C | 0.25 | +0.25 | -0.25 | 0.175 |
| D | 0.25 | +0.25 | -0.25 | 0.175 |

一步之后，p(B) 从 0.25 升到约 0.475，其余三个降到约 0.175。上面代码里的数值差分会验证这组梯度。

为什么这样设计。交叉熵衡量"模型给示范文本分配的总概率的负对数"，把它压到最低，等价于让示范的概率最大。注意 SFT 只推高 B：它看不到"B 和 D 其实都正确"这类额外信息，也看不到任何好坏判断。示范里没写过的内容，模型既不会学，也无从知道它好不好。这是 SFT 能力的上限，也是下一阶段引入偏好的直接原因。



RLHF 的训练分两步。第一步用标注排序训练 RM：对同一指令的一组候选，用成对排序损失让被偏好的回复得分更高。第二步把 RM 分数当作奖励优化模型，同时用 KL 项约束策略不要偏离参考策略太远。整体要最小化的目标写作

$$L_{\mathrm{RLHF}} = -\mathbb{E}_{y\sim\pi}\big[r_\theta(x,y)\big] + \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\mathrm{ref}}\big).$$

RM 分数高的候选被推高，KL 项把它拉回参考策略，防止模型钻 RM 的空子走得太远。


In [ ]:
import numpy as np

rm_scores = np.array([0.5, 1.0, -0.2, 0.0])
beta = 0.5
ref = np.full(4, 0.25)  # 参考策略：均匀分布


def rlhf_loss(theta):
    """RLHF 目标：负期望 RM 分数 + KL 约束。"""
    p = softmax(theta)
    reward = rm_scores @ p
    kl = (p * (np.log(p) - np.log(ref))).sum()
    return -reward + beta * kl


theta = np.zeros(4)
print("logit θ(B) | p(B)   | E[r]   | β·KL   | RLHF 损失")
for tb in [-1.5, -0.5, 0.0, 0.5, 1.5]:
    th = theta.copy()
    th[1] = tb
    p = softmax(th)
    kl = (p * (np.log(p) - np.log(ref))).sum()
    print(f"  {tb:+.1f}   | {p[1]:.3f} | {rm_scores@p:.3f} | {beta * kl:.3f} | {rlhf_loss(th):.4f}")

eps = 1e-4
grad = np.array([(rlhf_loss(theta + eps * np.eye(4)[j]) -
                  rlhf_loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                 for j in range(4)])
print()
print("RLHF 梯度 dL/dθ :", np.round(grad, 3))
print("关键观察：RM 分数高于均值(0.325)的 token 被推高，低于均值的被压低；KL 项把整体拉回均匀。")


RLHF 分两步，每一步都算一遍。

第一步训练 RM。给同一指令的一组候选，标注员挑出更被偏好的那个。训练 RM 用成对排序损失：对一对候选 (y_w, y_l)，要求被偏好的 y_w 得分明显高于 y_l，损失是

$$L_{\mathrm{RM}} = -\log\sigma\big(r(x, y_w) - r(x, y_l)\big).$$

拿玩具里的 RM 分数试两对。(B, C)：1.0 - (-0.2) = 1.2，σ(1.2) ≈ 0.769，损失 = -log 0.769 ≈ 0.263。(A, D)：0.5 - 0.0 = 0.5，σ(0.5) ≈ 0.622，损失 ≈ 0.474。得分差越大，损失越小——RM 被训练成"给更被偏好的回复打更高分"。

第二步把 RM 分数当奖励优化策略，目标是

$$L_{\mathrm{RLHF}} = -\mathbb{E}_{y\sim\pi}\big[r(x,y)\big] + \beta\, D_{\mathrm{KL}}\big(\pi \,\|\, \pi_{\mathrm{ref}}\big).$$

第一项是期望 RM 分数的相反数：概率越流向高分 token，这一项越小。第二项是 KL 散度，惩罚策略偏离参考策略 π_ref；模型偏离越多，KL 越大，β 控制惩罚强度。它防止模型为了刷 RM 分数而跑得太远。

手算初始点的梯度。π_ref 是均匀分布，初始 p 也是均匀分布，所以 KL = 0。期望 RM 分数 E[r] = (0.5 + 1.0 - 0.2 + 0.0)/4 = 0.325。对 L 求导并代入 p = 0.25，得到

$$\frac{\partial L}{\partial \theta_k} = -p_k\big(r_k - \mathbb{E}[r]\big).$$

| token | r_k | r_k - 0.325 | 梯度 | 更新后 p_k |
|:---|:---|:---|:---|:---|
| A | 0.5 | +0.175 | -0.044 | 0.259 |
| B | 1.0 | +0.675 | -0.169 | 0.294 |
| C | -0.2 | -0.525 | +0.131 | 0.218 |
| D | 0.0 | -0.325 | +0.081 | 0.229 |

梯度为负的 A、B 被推高，为正的 C、D 被压低。走一步后重新 softmax，B 的概率升到 0.294 最高，A 0.259 次之，C 0.218 最低。上面代码的扫描表和数值差分验证这组数字。

与 SFT 的差别在这里已经清楚。SFT 的信号是"示范答案是 B"，只有 B 被推高；RLHF 的信号是"四个候选的相对偏好"，整个分布按 RM 分数重新分配。SFT 学的是模仿示范，RLHF 学的是按偏好排序——模型会主动压低低分答案，哪怕它也在示范里出现过。



RLHF 的 RM 是代理目标，模型可能骗过它。如果 RM 只统计回复里是否出现某个关键词，模型就会堆砌关键词拿高分，而这些文本对用户没有价值——这是 `reward hacking`。RM 还只能衡量文本质量，衡量不了"这个动作序列能不能完成任务"。Constitutional AI 曾尝试压缩人工标注成本，让 AI 依据原则列表自我批评与修订，04 讲已精读，这里不再展开。

这两条限制是下一阶段更换信号来源的直接动机：与其学一个奖励函数，不如检查答案对不对。


RLHF 的两条限制，各用一个具体例子看清楚。

第一条，RM 是代理目标，可能被模型骗过。假设 RM 的训练数据里高分回复普遍含"首先""其次""因此"，RM 就会把"出现这类词"当成好回复的信号。模型发现这一点后会堆砌这些词来刷分数，而文本对用户没有实际价值。模型优化的是"让 RM 打高分"，不是"真正完成任务"，这个现象叫 reward hacking。

第二条，RM 只能衡量文本质量，衡量不了任务成败。让模型解数学题：RM 看到"过程完整、语气自信"的回复可能给高分，但答案如果是错的，这个高分就是错信号。标准答案不在 RM 的输入里，RM 无从核对。

两条都指向同一个方向：与其让模型猜"人喜欢什么"，不如直接告诉它"什么是对的"。能写判据的任务——答案有标准、代码有测试——可以直接检查。下一节的 RLVR 就是把这条思路做成训练目标。



## 3. Agent 化后训练：工具、执行与反馈

上一节的 RLHF 靠标注员打分，人既贵，RM 又可能被模型骗过。这一节解决一个问题：当任务有明确答案时，能不能不靠人打分。答案是能，用规则判断对错。这一节先讲用规则当奖励的方法，再一步步走到"让环境当裁判"的 Agent 后训练。

RLHF 的奖励模型是学出来的，模型可能骗过它。当任务能写出判据时，可以不要这个学出来的奖励模型，改用规则直接判断：答案和标准答案一致给 1，不一致给 0；代码通过全部测试给 1，否则给 0。这种用可验证奖励做强化学习的方法叫 `RLVR`（用可验证奖励做强化学习）。DeepSeek-R1 的 R1-Zero 从 base model 出发，不做任何 SFT，纯用 RLVR 训练，长链推理自发涌现——模型自己学会先想再答，因为想得越多越容易拿分。奖励只要对、且可验证，聪明的策略不需要人示范。

RLVR 的优化用组内 advantage：同一指令采样一组候选，把每个候选的奖励减去组均值、除以组标准差。这就是 `GRPO` 的核心，它不需要价值网络。下面先手算这组归一化 advantage，再把三种时代的损失形态并排对比，看它们各自把概率推向哪里。

RLVR 的奖励是一个判据，不是神经网络。判据的例子：答案与标准答案一致，给 1，否则给 0；代码通过全部测试，给 1，否则给 0。判据写好一次，就能对任意候选免费打分，这比训练 RM 便宜得多。

GRPO 用一个技巧把"奖励高低"转成"组内相对高低"。对同一指令采样一组候选，把每个奖励减去组均值、除以组标准差，得到组内 advantage。手算一组，设某指令采样 4 个候选，奖励为 [1, -1, 0, 1]：

组均值 = (1 + (-1) + 0 + 1)/4 = 0.25
偏差 = (0.75, -1.25, -0.25, 0.75)
方差 = (0.75² + 1.25² + 0.25² + 0.75²)/4 = 2.75/4 = 0.6875
标准差 = √0.6875 ≈ 0.829
advantage = 偏差/0.829 ≈ (0.905, -1.508, -0.302, 0.905)

奖励最高的候选拿到约 +0.9，最低的拿到约 -1.5。下面代码原样算出这组数。

为什么要减均值、除标准差。减均值挑出"这一组里谁高于平均"——高于平均给正 advantage，低于平均给负 advantage；除标准差做尺度统一，让不同组、不同奖励量纲的信号可比。模型学到的是"组内谁更好"，不是"谁的绝对分高"。代价也在这里：一组全对或全错时，均值就是 1 或 -1，减均值后每个 advantage 都是 0，这组样本不产生任何梯度。



In [ ]:
import numpy as np


def group_advantage(rewards, eps=1e-9):
    """组内归一化 advantage；(r - mean) / (std + eps) 防全对/全错组除零。"""
    return (rewards - rewards.mean()) / (rewards.std() + eps)


rewards = np.array([1.0, -1.0, 0.0, 1.0])
mean = rewards.mean()
std = rewards.std()
advantage = group_advantage(rewards)

print("奖励 r          :", rewards)
print("组均值          :", round(mean, 3))
print("组标准差        :", round(std, 3))
print("组内 advantage  :", np.round(advantage, 3))
print("关键观察：正确候选(1.0)获得正 advantage 被推高，错误候选获得负 advantage 被压低。")

# GRPO 目标随正确 token 概率的变化
print()
print("p(D) | GRPO 加权对数似然目标")
for p in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pi = np.array([(1 - p) / 3] * 3 + [p])
    adv = group_advantage(np.array([0.0, 1.0, 0.0, 1.0]))
    obj = -(adv * np.log(pi)).sum() / 4
    print(f" {p:.1f} |        {obj:.4f}")

# 全对组与全错组：advantage 恒为 0
print()
for name, r in [("全对组", np.array([1.0, 1.0, 1.0, 1.0])),
                ("全错组", np.array([-1.0, -1.0, -1.0, -1.0]))]:
    adv = group_advantage(r)
    print(f"{name} r={r.tolist()} -> advantage={np.round(adv, 3)}")
print("关键观察：组内无差异时 advantage 全为 0，这一组样本不产生任何梯度。")


现在把三种目标放在同一批数据上并排手算。数据就是前面的 toy：4 个 token A、B、C、D，初始概率全为 0.25。三份信号分别是：SFT 的示范 token 是 B；RLHF 的 RM 分数 [0.5, 1.0, -0.2, 0.0]；RLVR 的正确性 [0, 1, 0, 1]（B 和 D 正确）。

三种损失的梯度公式前面已各自求出：

SFT：  ∂L/∂θ_j = p_j - 1{j = 示范}
RLHF： ∂L/∂θ_k = -p_k (r_k - E[r])
RLVR： ∂L/∂θ_j = -advantage_j / 4

代入初始 p = 0.25，三组梯度是：

| 目标 | 梯度 dL/dθ (A, B, C, D) | 谁被推高 |
|:---|:---|:---|
| SFT | (+0.25, -0.75, +0.25, +0.25) | 只有示范 B |
| RLHF | (-0.04, -0.17, +0.13, +0.08) | 高于平均分的 A、B |
| RLVR | (+0.25, -0.25, +0.25, -0.25) | 正确的 B、D |

每组用学习率 1 走一步，再 softmax，同一个初始点出发的三份模型，一步之后概率变成：

| 目标 | 一步后概率 (A, B, C, D) |
|:---|:---|
| SFT | (0.175, 0.475, 0.175, 0.175) |
| RLHF | (0.259, 0.294, 0.218, 0.229) |
| RLVR | (0.189, 0.311, 0.189, 0.311) |

三个模型看同一份数据，走出不同方向，差异全来自信号。

最值得注意的分歧在 token D。RLVR 认为 D 正确（正确性数组里 D 是 1），把 D 的概率抬到与 B 并列；RLHF 里人给 D 打了 0 分，低于平均 0.325，于是 RLHF 反而压低 D。同一个 token，客观上是正确答案，却不讨人喜欢——两种信号给出相反的更新方向。

把三行合起来看，就能说清"信号来源迁移到底迁移了什么"。SFT 认识的是示范里写了什么，只推 B；RLHF 认识的是人觉得谁更好，按分数重排整个分布；RLVR 认识的是客观对不对，所有正确的都推，不管人喜不喜欢。迁移的是模型去优化的信号：从"一个正确答案"，到"相对偏好"，再到"客观对错"。下面代码用数值差分验证这三组梯度数字。



In [ ]:
import numpy as np

demo_idx = 1
rm_scores = np.array([0.5, 1.0, -0.2, 0.0])
correctness = np.array([0.0, 1.0, 0.0, 1.0])
beta = 0.5
ref = np.full(4, 0.25)


def sft_loss(theta):
    """SFT 交叉熵。"""
    return -np.log(softmax(theta)[demo_idx])


def rlhf_loss(theta):
    """RLHF 目标：负期望 RM 分数 + KL 约束。"""
    p = softmax(theta)
    return -(rm_scores @ p) + beta * (p * (np.log(p) - np.log(ref))).sum()


def grpo_loss(theta):
    """组内 advantage 加权的对数似然（策略梯度的等价目标）。"""
    p = softmax(theta)
    adv = group_advantage(correctness)
    return -(adv @ np.log(p)).sum() / len(correctness)


def grad_of(loss, theta):
    """数值差分求梯度。"""
    eps = 1e-4
    return np.array([(loss(theta + eps * np.eye(4)[j]) -
                      loss(theta - eps * np.eye(4)[j])) / (2 * eps)
                     for j in range(4)])


theta = np.zeros(4)
for name, loss in [("SFT", sft_loss), ("RLHF", rlhf_loss), ("RLVR/GRPO", grpo_loss)]:
    g = grad_of(loss, theta)
    direction = ["↑推高" if x < 0 else "↓压低" for x in g]
    print(f"{name:9s} 梯度 {np.round(g, 3)}  方向 {direction}")
print()
print("关键观察：SFT 只推高示范 token B；RLHF 推高 RM 分数高的 A、B；RLVR 推高正确的 B、D。")


代码打印的三组梯度与手算一致：SFT 只推 B；RLHF 推 A、B，压 C、D；RLVR 推 B、D，压 A、C。同一个初始模型、同一份数据，三项目标给出的更新方向不同——梯度就是目标函数告诉模型的"该往哪走"。

这个静态对比只看了一步。真实的训练是动态的：每更新一次，模型重新采样，判据或环境重新打分，下一批数据又来了。SFT 看到的是固定的示范集，RLHF 看到的是固定的 RM 分数，RLVR 看到的是随采样变化的一组奖励。要看收敛行为，必须真跑训练循环。下一节在同一个 toy bandit 上实现 PPO 与 GRPO，比较两种 advantage 估计方式在动态更新下的表现。



上面的对比是静态的：给定一套概率看梯度方向。真实的训练是动态的——每更新一步，模型重新采样、环境重新打分。下面在 toy bandit 上分别实现 `PPO`（近端策略优化）与 GRPO，观察两种 advantage 估计方式如何影响收敛。

PPO 是当前训练聊天模型用得最广的强化学习方法。它每一步更新时都限制新策略不要偏离旧策略太远，避免一次更新把模型搞坏，所以叫"近端"。GRPO 是 06 讲读过的 DeepSeek 系列使用的方法，它省掉了 PPO 里的价值网络。

设定是一个上下文 bandit：5 个问题，每个问题有唯一正确的动作（共 10 个候选）。环境按规则判定动作对错，奖励 +1/-1，没有可 hack 的中间层。PPO 用一个价值网络估计基线，GRPO 用组内均值做基线。两者都要回答同一件事：让每个问题的正确动作概率升上去。

PPO 和 GRPO 都基于同一个策略梯度更新：采样到动作 a 后，把 a 的 logit 往 advantage 高的方向推。写成式子：

$$\logits[a] \leftarrow \logits[a] + \alpha A (1 - p(a)), \qquad \logits[j \neq a] \leftarrow \logits[j] - \alpha A\, p(j).$$

A 是 advantage。A 为正，动作 a 的 logit 上升、其他动作的 logit 下降；A 为负则相反。一次采样只给出一条 (问题, 动作, 奖励)，A 就是这条给策略的全部信号。

PPO 与 GRPO 的差别只在 A 从哪来。PPO 用价值网络：每个问题维护一个标量基线 value[q]，A = r - value[q]。value[q] 在训练中不断向"该问题的期望奖励"回归。本 toy 里每个问题均匀采样时选对的概率是 1/10，所以 value[q] 收敛到 1×(1/10) + (-1)×(9/10) = -0.8；于是正确动作的 advantage 是 1 - (-0.8) = 1.8，错误动作是 -1 - (-0.8) = -0.2。advantage 衡量的是"这次实际奖励比平时好多少"。

GRPO 不用价值网络，A 由同一组内其他候选的奖励算出来：A_i = (r_i - 组均值)/组标准差。省掉价值网络这个额外网络，代价是每组要采样多个候选，而且全对/全错组的 advantage 全为 0，梯度落空。

容易混淆的一点：advantage 和原始奖励 r 的平均值不同。r 的平均值表示"这组数据平均拿多少分"；advantage 的平均值恒为 0，因为减均值把中心移到了组内平均。策略梯度关心的是相对高低，不是绝对分数。



In [ ]:
import numpy as np

rng = np.random.default_rng(42)

NUM_Q = 5
NUM_A = 10
correct = rng.integers(0, NUM_A, size=NUM_Q)   # 每个问题的正确动作

logits = np.zeros((NUM_Q, NUM_A))   # 策略参数
value = np.zeros(NUM_Q)             # 价值网络：每个问题一个标量基线
lr_policy = 0.05
lr_value = 0.1


def sample_one():
    """采样一条 (问题, 动作)，返回问题、动作、奖励与采样概率。"""
    q = int(rng.integers(0, NUM_Q))
    p = softmax(logits[q])
    a = int(rng.choice(NUM_A, p=p))
    r = 1.0 if a == correct[q] else -1.0
    return q, a, r, p


def ppo_step():
    """PPO 简化版：advantage = 奖励 - 价值基线，策略按 advantage 加权更新。"""
    for _ in range(32):
        q, a, r, p = sample_one()
        adv = r - value[q]
        logits[q] += lr_policy * adv * (np.eye(NUM_A)[a] - p)
        value[q] += lr_value * (r - value[q])   # 价值向奖励回归


ppl_acc = []
for it in range(80):
    ppo_step()
    p = softmax(logits)
    acc = (np.argmax(p, axis=1) == correct).mean()
    ppl_acc.append(acc)

print("PPO 正确率（每 20 轮）:", [round(float(x), 2) for x in ppl_acc[::20]])
print("PPO 末期每个问题正确动作的概率:",
      np.round(softmax(logits)[np.arange(NUM_Q), correct], 2))


In [ ]:
import numpy as np

rng = np.random.default_rng(7)

# 与 PPO 同一组问题，重新初始化策略
logits_g = np.zeros((NUM_Q, NUM_A))
lr_g = 0.2
K = 4  # 每组采样 4 个动作


def grpo_update(q):
    """GRPO：对一个问题采样一组动作，组内归一化 advantage，无价值网络。"""
    p = softmax(logits_g[q])
    acts = rng.choice(NUM_A, size=K, p=p)
    rewards = np.array([1.0 if a == correct[q] else -1.0 for a in acts])
    adv = group_advantage(rewards)
    for i, a in enumerate(acts):
        onehot = np.zeros(NUM_A)
        onehot[a] = 1.0
        logits_g[q] += lr_g / K * adv[i] * (onehot - p)
    return rewards


g_acc = []
wasted_history = []
for it in range(80):
    wasted = 0
    for _ in range(8):   # 每轮 8 个组
        q = int(rng.integers(0, NUM_Q))
        rewards = grpo_update(q)
        if rewards.max() == rewards.min():
            wasted += 1
    wasted_history.append(wasted)
    p = softmax(logits_g)
    g_acc.append((np.argmax(p, axis=1) == correct).mean())

print("GRPO 正确率（每 20 轮）:", [round(float(x), 2) for x in g_acc[::20]])
print("每轮全对/全错组数量（前 10 轮）:", wasted_history[:10])
print("关键观察：GRPO 不需要价值网络，但全对组与全错组不产生任何梯度，这批样本白算。")

import matplotlib.pyplot as plt

plt.figure(figsize=(6.2, 3.8))
plt.plot(ppl_acc, label="PPO (critic baseline)")
plt.plot(g_acc, label="GRPO (group baseline)")
plt.xlabel("round")
plt.ylabel("accuracy")
plt.title("Correct-arm accuracy on a toy bandit")
plt.legend()
plt.tight_layout()
plt.show()


RLVR 解决了"文本好不好"的问题，但它只适用于能写判据的任务——数学、代码、谜题。开放问题（写一封得体的邮件）没有标准答案，也就没有验证器。Agent 化后训练的关键转移是：任务虽然开放，但环境本身可以当验证器。任务成功可以用测试是否通过、目标是否达成、终端状态是否收敛来判定。

训练单位也从一段文本变成一整条轨迹。动作空间从下一个 token 扩展成工具调用序列：查数据库、执行代码、写文件、返回结果。奖励不再是任务开始前写死的判据，而是环境跑出来的结果——可能稀疏，也可能延迟到轨迹末端才出现。下面先把四个阶段的信号来源放在同一张图上，看成本与可靠性如何一路变化。


Agent 后训练最关键的一句是：任务虽然开放，环境本身可以当验证器。拿具体任务看这句话。

写一封得体的邮件是开放任务，没有标准答案，写不了判据。但"用搜索完成一份报告并输出文件"就可以验证：文件是否生成、格式是否达标、关键数据是否出现，甚至测试脚本能否跑通。开放任务的"成功"由执行结果判定，不由人预设。

训练单位随之改变。SFT 到 RLVR 的样本都是文本——一个回复，或一道题的答案。Agent 后训练要学的是动作序列，样本是一条轨迹：调用搜索、读返回、执行代码、写文件、返回结果。动作空间从"下一个 token"扩展成"工具调用序列"。

信号变稀疏、变延迟。判据奖励在动作结束时立刻给分；Agent 的奖励要等整条轨迹跑完，环境才知道成败——可能是几十步之后的一个 +1 或 -1。下一张图把四个阶段的信号投影到（标注成本，抗 hack 可靠性）平面，看迁移的方向。



In [ ]:
import matplotlib.pyplot as plt

# 四个信号来源，投影到 (标注成本, 抗 hack 可靠性) 平面
signals = [
    ("SFT human demo", 1.0, 0.30),
    ("RLHF human pref", 0.8, 0.50),
    ("RLVR rule", 0.4, 0.90),
    ("Agent RL env", 0.2, 1.00),
]
names = [s[0] for s in signals]
costs = [s[1] for s in signals]
rel = [s[2] for s in signals]

plt.figure(figsize=(6.2, 4.4))
plt.scatter(costs, rel, s=260, c=range(4), cmap="viridis")
for i, name in enumerate(names):
    plt.annotate(name, (costs[i], rel[i]),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
plt.xlabel("annotation cost (lower is cheaper)")
plt.ylabel("resistance to reward hacking")
plt.title("Reward signal sources across post-training stages")
plt.xlim(0, 1.2)
plt.ylim(0, 1.2)
plt.tight_layout()
plt.show()

print("关键观察：信号来源从右上角迁向左下角——标注成本一路下降，可靠性一路上升。")


轨迹级奖励把整条轨迹压缩成一个 +1 或 -1。如果一条轨迹有几十步，只有最后一步给信号，模型无法判断中间哪一步错了——这是 `credit assignment` 问题，也是稀疏延迟奖励的核心困难。步级（里程碑）奖励把总奖励拆到每一步：每达成一个子目标就给一份，末尾成功再给一份，模型因此收到过程信号。

下面在一个 toy 三步环境里数值对比两种奖励的梯度方差与收敛速度。轨迹奖励在每一步之间分摊同一个终局值，方差被放大；里程碑奖励每步只携带自己的子目标信号，方差更小。


把 credit assignment 问题用具体数字拆开。三步任务（查数据库 → 过滤 → 返回结果），每步从 3 个动作里选，三步全对才成功。

先算成功率。均匀策略下每步选对的概率是 1/3，三步全对的概率是 (1/3)³ = 1/27 ≈ 0.037。轨迹奖励下，每 27 条轨迹大约只有 1 条拿到 +1，其余 26 条都是 -1。一条轨迹如果第一步做对了、后面错了，它和"三步全错"拿到的信号一样是 -1，模型无从分辨"我第一步其实做对了"。

从梯度看更清楚。策略梯度的每步更新量是

$$g_t = R\,(\mathrm{onehot}(a_t) - p_t),$$

R 是这条轨迹的回报。轨迹奖励下，同一个 R（+1 或 -1）同时缩放三步的梯度；里程碑奖励把 R 换成每步的子目标信号 r_t（本 toy 每步 +0.3，末尾成功再 +0.1），第一步做对就单独收到 +0.3，与后面成败无关。

方差的差异可以定量。均匀策略下，单看梯度的一个分量：轨迹奖励的 R² = 1，梯度方差 ≈ 2/9 ≈ 0.22；里程碑奖励只有选对的那一步携带信号，梯度方差 ≈ 0.007。前者约为后者的 30 倍。方差大意味着同样采样数下梯度方向波动大、收敛慢；里程碑把信号按步拆开，每步都有确定的过程信号。下面代码在同一环境实测两种奖励的梯度方差与收敛曲线。



In [ ]:
import numpy as np

rng = np.random.default_rng(1)
N_ACT = 3
correct_steps = np.array([0, 1, 2])   # 第 3 步的正确选择决定终局成败


def sample_trajectory(logits):
    """按策略逐步采样三步动作。"""
    return np.array([rng.choice(N_ACT, p=softmax(logits[t])) for t in range(3)])


def trajectory_reward(acts):
    """只有三步全对给 +1，否则 -1。"""
    return 1.0 if (acts == correct_steps).all() else -1.0


def milestone_rewards(acts):
    """每步子目标达成 +0.3，末尾成功再 +0.1。"""
    per = 0.3 * (acts == correct_steps).astype(float)
    if (acts == correct_steps).all():
        per[2] += 0.1
    return per


def reinforce_grad(logits, mode):
    """一次 episode 的策略梯度估计。"""
    acts = sample_trajectory(logits)
    g = np.zeros_like(logits)
    if mode == "trajectory":
        R = trajectory_reward(acts)
        for t in range(3):
            g[t] = R * (np.eye(N_ACT)[acts[t]] - softmax(logits[t]))
    else:
        r = milestone_rewards(acts)
        for t in range(3):
            g[t] = r[t] * (np.eye(N_ACT)[acts[t]] - softmax(logits[t]))
    return g, acts


# 从同一均匀策略出发，各采样 500 次梯度，比较方差
logits0 = np.zeros((3, N_ACT))
traj_grads, mile_grads = [], []
for _ in range(500):
    g1, _ = reinforce_grad(logits0, "trajectory")
    g2, _ = reinforce_grad(logits0, "milestone")
    traj_grads.append(g1)
    mile_grads.append(g2)
var_t = np.array(traj_grads).var()
var_m = np.array(mile_grads).var()
print("均匀策略下梯度方差：轨迹奖励", round(var_t, 4), " vs 里程碑奖励", round(var_m, 4))
print()

# 训练对比：400 轮，画 50 轮滑动平均的回报
M = 400
lr = 0.1


def train(mode):
    """跑 M 轮 REINFORCE，返回逐轮终局回报。"""
    logits = np.zeros((3, N_ACT))
    returns = []
    for it in range(M):
        g, acts = reinforce_grad(logits, mode)
        logits += lr * g
        returns.append(1.0 if (acts == correct_steps).all() else -1.0)
    return returns


def running_mean(x, w=50):
    """w 窗口的滑动平均。"""
    out = []
    for i in range(len(x)):
        lo = max(0, i - w + 1)
        out.append(np.mean(x[lo:i + 1]))
    return out


ret_t = running_mean(train("trajectory"))
ret_m = running_mean(train("milestone"))
print("400 轮末的滑动平均回报：轨迹奖励", round(ret_t[-1], 3),
      " vs 里程碑奖励", round(ret_m[-1], 3))

import matplotlib.pyplot as plt

plt.figure(figsize=(6.2, 3.8))
plt.plot(ret_t, label="trajectory reward")
plt.plot(ret_m, label="milestone reward")
plt.xlabel("episode")
plt.ylabel("running mean return")
plt.title("Credit assignment: sparse vs dense reward")
plt.legend()
plt.tight_layout()
plt.show()


代码的测量印证了手算：均匀策略下轨迹奖励的梯度方差 0.2222，里程碑奖励只有 0.0072，相差约 30 倍；训练 400 轮后，轨迹奖励的滑动平均回报停在约 0.16，里程碑奖励升到约 0.52。信号在每一步都可用的模型，收敛得更快。

credit assignment（信用分配）这个术语，指"把整条轨迹的成功或失败，归因到其中某一步"。轨迹奖励的归因是"所有步一起担责"：终局失败，每一步的梯度都被同一个负号缩放，哪怕某一步做对了。里程碑奖励的归因是"每步只对自己的子目标负责"：第一步做对就收 +0.3，与后面成败无关。归因越精细，梯度信号越不混乱。

Agent 任务的轨迹动辄几十步，奖励天然稀疏——环境只在任务结束时给信号。MiRA 的做法是把"终局成败"拆成若干可判定的里程碑，让模型每一步都有过程信号可学。



把整条链路合起来，就是一个环境反馈飞轮。模型输出动作序列 → 环境执行并判定成败 → 成败信号当作奖励更新策略 → 过滤成功的轨迹、重新采样。这就是 STaR 与 WebRL 的骨架。失败的任务有两种命运：直接丢弃，或改造成可验证的形式重新注入训练集，后者让数据池随训练逐轮膨胀。

下面用一个 toy 算术任务把这个飞轮跑起来。模型（llm_client 的 脚本化示例 实例）对每个问题提出答案，环境核对答案并返回 +1/-1，我们据此更新一个轻量的答案策略。每轮统计成功率，观察它逐轮上升，同时演示失败任务的两种命运。


In [ ]:
import sys, os, re
import numpy as np

# 统一走仓库根目录的 llm_client.py，真实 API 演示下同样可跑
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, "llm_client.py")):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm

client = get_llm()
print("当前 LLM 模式:", "脚本化示例" if False else "real API")

# 问题池：前三题 脚本化示例 能直接算出加法，最后一题乘法 脚本化示例 不会处理
questions = ["计算 13 加 24 等于几", "计算 35 加 19 等于几",
             "计算 7 加 46 等于几", "计算 12 乘 7 等于几"]
answers = {"计算 13 加 24 等于几": 37, "计算 35 加 19 等于几": 54,
           "计算 7 加 46 等于几": 53, "计算 12 乘 7 等于几": 84}


def extract_number(text):
    """从回复里提取第一个整数；提取不到返回 None。"""
    m = re.search(r"-?\d+", text)
    return int(m.group()) if m else None


def propose(q):
    """模型提出答案：问一次 LLM，解析出数字。"""
    reply = client.chat([{"role": "user", "content": f"{q}，只输出数字。"}])
    return extract_number(reply)


proposal = {q: propose(q) for q in questions}

# 候选答案池：脚本化示例 的提案 + 干扰项；乘法题的正确答案不在池里
candidate_pool = {}
for q in questions:
    cands = []
    if proposal[q] is not None:
        cands.append(proposal[q])
    cands += [answers[q] + 3, answers[q] - 7]   # 干扰项
    candidate_pool[q] = list(dict.fromkeys(cands))

for q in questions:
    print(f"{q}: 提案 {proposal[q]}, 候选池 {candidate_pool[q]}")
print("关键观察：前三题的提案就是正确答案；乘法题的提案是 None，正确答案不在池中。")


In [ ]:
rng = np.random.default_rng(3)
lr = 0.2
logits_map = {q: np.zeros(len(candidate_pool[q])) for q in questions}
q_fail = questions[-1]


def run_round():
    """跑一轮 RL：每个问题采样答案，环境核对，REINFORCE 更新。返回逐题是否成功。"""
    per_q = {}
    for q, cands in candidate_pool.items():
        p = softmax(logits_map[q])
        idx = int(rng.choice(len(cands), p=p))
        a = cands[idx]
        r = 1.0 if a == answers[q] else -1.0
        logits_map[q] += lr * r * (np.eye(len(cands))[idx] - p)
        per_q[q] = (a == answers[q])
    return per_q


def correct_prob():
    """正确答案的平均策略概率（只统计正确答案在池里的问题）。"""
    vals = []
    for q, cands in candidate_pool.items():
        if answers[q] in cands:
            vals.append(float(softmax(logits_map[q])[cands.index(answers[q])]))
    return np.mean(vals)


def block_means(x):
    """把逐轮序列切成每 15 轮一块，返回块均值。"""
    return [round(float(np.mean(x[i * 15:(i + 1) * 15])), 2)
            for i in range(len(x) // 15)]


# 不改造失败任务：连续 60 轮，观察成功率与正确答案概率
rates, probs, fail_rate = [], [], []
for it in range(60):
    per_q = run_round()
    rates.append(np.mean(list(per_q.values())))
    probs.append(correct_prob())
    fail_rate.append(per_q[q_fail])

print("整体采样成功率（每 15 轮）:", block_means(rates))
print("正确答案平均概率（每 15 轮）:", block_means(probs))
print("乘法问题采样成功率（每 15 轮）:", block_means(fail_rate))
print("关键观察：前 3 题正确答案概率逐块上升；乘法题的正确答案不在池中，成功率恒为 0——任务被丢弃。")


注意上面乘法问题的结果：采样成功率恒为 0。原因不是模型不会，而是正确动作根本不在候选池里。

RL 的更新只能调整"池子里已有动作"的概率。乘法问题的候选池是 [87, 77]（脚本化示例 的提案是 None，被跳过了），正确答案 84 不在其中。策略不管怎么调，都只能在这两个里分配概率，永远选不中 84。这相当于动作空间里根本没有"正确动作"这个选项，再好的奖励也推不到它。

处理失败任务有两种做法。第一种是丢弃：从训练集里拿掉，模型在这个问题上永远学不会。第二种是改造：把任务拆成环境能验证的子步骤，让模型逐步核对，把验证出的正确答案补进候选池。改造让失败任务重新变得可训练。下面代码演示改造后的过程——候选池膨胀，正确概率逐块上升。



In [ ]:
import matplotlib.pyplot as plt

# 命运 B：改造——把乘法拆成连加，用 脚本化示例 逐个核对子目标，环境验证终值
acc = 12
verified = None
for step in range(6):
    target = acc + 12
    reply = client.chat([{"role": "user",
                          "content": f"检查 {acc} 加 12 是否等于 {target}，只输出数字。"}])
    if extract_number(reply) != target:
        break
    acc = target
verified = acc

pool_before = sum(len(c) for c in candidate_pool.values())
print("改造得到的验证答案:", verified)
assert verified is not None, "真实模型没有返回可验证的答案"

# 正确答案进入候选池，策略重新初始化
candidate_pool[q_fail] = list(dict.fromkeys(candidate_pool[q_fail] + [verified]))
logits_map[q_fail] = np.zeros(len(candidate_pool[q_fail]))
pool_after = sum(len(c) for c in candidate_pool.values())
print("改造后乘法问题候选池:", candidate_pool[q_fail])
print("数据池大小: 改造前", pool_before, "→ 改造后", pool_after)

# 继续跑 100 轮 RL，观察乘法问题随数据池膨胀而学会
rates2, prob2, fail2 = [], [], []
for it in range(100):
    per_q = run_round()
    rates2.append(np.mean(list(per_q.values())))
    prob2.append(float(softmax(logits_map[q_fail])[candidate_pool[q_fail].index(84)]))
    fail2.append(per_q[q_fail])

print("改造后整体采样成功率（每 20 轮）:", block_means(rates2))
print("改造后乘法问题正确概率（每 20 轮）:", block_means(prob2))
print("改造后乘法问题采样成功率（每 20 轮）:", block_means(fail2))

plt.figure(figsize=(6.2, 3.8))
plt.plot(rates, label="overall (before curriculum)")
plt.plot(rates2, label="overall (after curriculum)")
plt.plot(prob2, label="correct prob of failed task")
plt.xlabel("round")
plt.ylabel("success rate / prob")
plt.title("Environment feedback flywheel")
plt.legend()
plt.tight_layout()
plt.show()


改造后的输出说明了三件事。第一，验证出的正确答案 84 进入候选池后，乘法问题的正确概率从首块均值 0.62 逐块升到 0.97，采样成功率随之升高——任务从"永远失败"变成"可学会"。第二，数据池从改造前的 11 个候选膨胀到改造后的 12 个，这就是"数据池随训练逐轮膨胀"：每解决一个失败任务，就往池子里补一组已验证数据。

这就是环境反馈飞轮的一个完整回合：模型输出动作序列 → 环境执行并判定成败 → 成败信号更新策略 → 过滤或改造失败任务 → 重新采样。飞轮每转一圈，可训练的数据多一分，模型对环境的把握多一分。把四个阶段的演进路线放到一起看，见下一节。



## 4. 演进路线图：信号来源的迁移

前三节把四种训练信号逐个讲了一遍，这一节把它们放到同一条时间线上，回答一个问题：这四种信号之间是什么关系。答案是一路的迁移。

2021-2022 的 SFT 用人类示范，2022-2023 的 RLHF 用人类偏好，2024-2025 的 RLVR 用规则判据，2024-2026 的 Agent 后训练用环境结果。每一个新阶段没有淘汰前面的，而是叠加在它之上——Agent 模型也要先 SFT、再对齐、再 RLVR，最后才做轨迹级 RL。

下图把各阶段的代表工作标在时间线上，信号来源写在节点下方。

In [ ]:
import matplotlib.pyplot as plt

stages = [
    (2021.0, "SFT\nhuman demos", "FLAN / InstructGPT-SFT"),
    (2022.6, "RLHF\nhuman prefs", "InstructGPT / ChatGPT"),
    (2024.1, "RLVR\nverifiable rules", "DeepSeek-R1 / DAPO"),
    (2025.3, "Agent RL\nenvironment", "RLEF / WebRL / MiRA"),
]

fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.axhline(0, color="gray", lw=1)
for x, label, work in stages:
    ax.scatter(x, 0, s=90, zorder=3)
    ax.annotate(label, (x, 0), xytext=(0, 14), textcoords="offset points",
                ha="center", fontsize=9)
    ax.annotate(work, (x, 0), xytext=(0, -20), textcoords="offset points",
                ha="center", fontsize=7, color="dimgray")
ax.set_xlim(2020, 2026.8)
ax.set_ylim(-0.4, 0.4)
ax.axis("off")
plt.title("Post-training evolution: reward moves from humans to environment")
plt.tight_layout()
plt.show()


把这条时间线映射回课程地图。04 讲的 RLEF 与 Constitutional AI 是 Agent 反馈的种子；06 讲的 GRPO 与 DAPO 是 Agent 后训练的算法引擎；08 讲的深度研究是"环境当验证器"的一种具体形态。本讲把它们收拢成一条主线。后面 13 讲的 SWE 智能体、14 讲的记忆、17 讲的评测，都会反复用到轨迹、环境反馈与评测这三个词。

一句话收束：后训练的历史，就是奖励信号从人类手里、交到验证器手里、最后交到环境手里的历史。


## 小结

这一讲所学的内容：

- [ ] 预训练模型是续写器，指令对它是待续写文本；后训练把优化目标从下一个 token 改成用户意图
- [ ] SFT 用人类示范，损失是交叉熵，只推高示范过的行为，无法超越示教者
- [ ] RLHF 用人类偏好训练 RM 再优化 RM 分数，加 KL 约束防走远；RM 是代理目标，可能被 reward hacking
- [ ] RLVR 用规则判据当奖励，GRPO 用组内 advantage 归一化，不需要价值网络
- [ ] 全对组与全错组的 advantage 全为 0，这批样本不产生梯度、算力白费
- [ ] PPO 用价值网络做基线，GRPO 用组均值做基线，两者收敛行为不同
- [ ] Agent 后训练把训练单位从文本升级为轨迹，奖励来自环境执行结果，可能稀疏且延迟
- [ ] 轨迹级奖励的梯度方差大、收敛慢；里程碑奖励提供过程信号，方差小
- [ ] 环境反馈飞轮：模型输出 → 环境判定 → 信号当奖励 → 过滤成功轨迹再采样；失败任务可丢弃或改造
- [ ] 四个阶段信号来源：人类示范 → 人类偏好 → 规则判据 → 环境执行结果，成本下降、更难被 hack


## 作业

> 可以让 AI 帮忙解释思路，但不建议直接让 AI "做完这道题"。


**作业 1：后训练四阶段分类**

下面 6 条训练设置描述分别属于哪个阶段。补全 classify 函数，让每条描述返回 "SFT" / "RLHF" / "RLVR" / "Agent RL"。参考答案已填好，请先在草稿上自己补全一遍，再运行对照。

小提示：先找描述里的信号来源——示范、偏好排序、规则判据、还是环境执行结果。每条描述只对应一个阶段。


In [ ]:
def classify(desc):
    """根据描述里的信号来源判断训练阶段。"""
    if "轨迹" in desc or "沙箱" in desc:
        return "Agent RL"
    if "标准答案" in desc or "规则函数" in desc or "通过测试" in desc:
        return "RLVR"
    if "示范" in desc or "期望回复" in desc:
        return "SFT"
    return "RLHF"


descs = [
    "用人工书写的 (指令, 期望回复) 做交叉熵微调",
    "让标注员对 6 个候选回复排序，训练奖励模型后做 PPO",
    "用规则函数判断答案与标准答案是否一致，做 GRPO",
    "在沙箱环境里跑完整 Agent 轨迹，用测试是否通过当奖励",
    "让 AI 依据原则列表自我批评与修订，用偏好做 PPO",
    "用规则函数判断代码是否通过测试，奖励 +1/-1",
]
stages = [classify(d) for d in descs]
print("分类结果：", stages)

assert stages[0] == "SFT"
assert stages[1] == "RLHF"
assert stages[2] == "RLVR"
assert stages[3] == "Agent RL"
assert stages[4] == "RLHF"
assert stages[5] == "RLVR"
print("6 条全部判对。抓住信号来源，就能定位训练阶段。")


**作业 2：组内 advantage 与零梯度组**

rewards 是一个组内奖励数组。补全 grpo_advantage，返回 (r - mean) / std；再补全 has_zero_signal，判断一组奖励是否产生零梯度（全对或全错）。参考答案已填好，请先在草稿上自己补全一遍，再运行对照。

小提示：先算均值、减均值、再除标准差；std 分母加一个微小量（如 1e-9），防止全对/全错组除零。


In [ ]:
import numpy as np


def grpo_advantage(rewards):
    """组内归一化 advantage；std 分母加微小量防除零。"""
    mean = rewards.mean()
    std = rewards.std() + 1e-9
    return (rewards - mean) / std


def has_zero_signal(rewards):
    """全对或全错时 advantage 全为 0，返回 True。"""
    return bool(rewards.max() == rewards.min())


r2 = np.array([1.0, 1.0, -1.0, -1.0])
a2 = grpo_advantage(r2)
print("r =", r2, "-> advantage =", np.round(a2, 3))
assert np.allclose(a2, [1.0, 1.0, -1.0, -1.0])
assert has_zero_signal(np.array([1.0, 1.0, 1.0])) is True
assert has_zero_signal(np.array([-1.0, -1.0])) is True
assert has_zero_signal(r2) is False
assert np.allclose(grpo_advantage(np.array([1.0, 1.0, 1.0])), 0.0)

print("advantage 归一化与零梯度组判定都正确。")
print("全对/全错组的 advantage 全为 0，不产生任何梯度；大量这样的组就是算力浪费，DAPO 用动态采样解决它。")


**作业 3：轨迹奖励与里程碑奖励**

toy 三步工具调用任务：查数据库 → 过滤 → 返回结果，每步有子目标（可判定是否达成）。补全 trajectory_reward（只有三步全成功给 +1）与 milestone_reward（每步子目标达成 +0.3，最后成功再 +0.1），并打印两种奖励对同一轨迹的差异。参考答案已填好，请先在草稿上自己补全一遍，再运行对照。

小提示：里程碑奖励其实是在给模型过程信号，这是 MiRA 的动机；先判断每步子目标是否达成，再累加。


In [ ]:
def trajectory_reward(steps_ok):
    """steps_ok 是三个布尔值。只有全部成功给 +1，否则 -1。"""
    return 1.0 if all(steps_ok) else -1.0


def milestone_reward(steps_ok):
    """每步子目标达成 +0.3，最后成功再 +0.1（合计 1.0），保留一位小数。"""
    per_step = 0.3 * sum(steps_ok)
    final = 0.1 if all(steps_ok) else 0.0
    return round(per_step + final, 1)


ok_all = [True, True, True]
ok_partial = [True, False, True]

assert trajectory_reward(ok_all) == 1.0
assert trajectory_reward(ok_partial) == -1.0
assert milestone_reward(ok_all) == 1.0
assert milestone_reward(ok_partial) == 0.6

print("轨迹奖励：全对", trajectory_reward(ok_all), "，部分对", trajectory_reward(ok_partial))
print("里程碑奖励：全对", milestone_reward(ok_all), "，部分对", milestone_reward(ok_partial))
print("同一轨迹，轨迹奖励只给 -1，里程碑奖励给了 +0.6 的过程信号——模型知道第一步做对了。")


## 参考资料

- Ouyang et al., [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155), 2022 — InstructGPT：SFT→RM→PPO 三段式 RLHF 的标杆，ChatGPT 的技术前身，1.3B 打平 175B 的参数差
- Christiano et al., [Deep Reinforcement Learning from Human Preferences](https://arxiv.org/abs/1706.03741), 2017 — RLHF 的思想源头：用人类偏好学奖励模型，而非手写奖励
- Bai et al., [Constitutional AI: Harmlessness from AI Feedback](https://arxiv.org/abs/2212.08073), 2022 — 用 AI 反馈压缩人工标注成本；04 讲已精读
- Wei et al., [Finetuned Language Models are Zero-Shot Learners](https://arxiv.org/abs/2109.01652), 2021 — FLAN：instruction tuning 的代表，SFT 时代泛化到新指令的证据
- Shao et al., [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/abs/2402.03300), 2024 — GRPO 的提出之处，组内 advantage 的思想来源；06 讲已精读
- Guo et al., [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948), 2025 — RLVR 的标志：R1-Zero 无 SFT 纯 RL、thinking 自发涌现；"没有可 hack 的 RM"原文出处
- Yu et al., [DAPO: An Open-Source LLM Reinforcement Learning System at Scale](https://arxiv.org/abs/2503.14476), 2025 — RLVR 的工程化：全对/全错组零梯度、动态采样与 Clip-Higher；06 讲已精读
- Chen et al., [RLEF: Grounding Code LLMs in Execution Feedback with Reinforcement Learning](https://arxiv.org/abs/2410.02089), 2024 — 执行反馈 + RL 教会模型修代码，Agent 轨迹 RL 的最小原型；04 讲已精读
- Yao et al., [WebShop: Towards Scalable Real-World Web Interaction with Grounded Language Agents](https://arxiv.org/abs/2207.01206), 2022 — 早期 IL+RL 网页购物 Agent，Agent 后训练的对照起点
- Qin et al., [ToolLLM: Facilitating Large Language Models to Master 16000+ Real-world APIs](https://arxiv.org/abs/2307.16789), 2023 — 工具调用训练的早期代表，把调用哪个 API 当作动作
- Xu et al., [WebRL: Training LLM Web Agents via Self-Evolving Online Curriculum Reinforcement Learning](https://arxiv.org/abs/2411.02337), 2024 — 网页 Agent 的自演化在线课程 RL；Llama-3.1-8B 在 WebArena-Lite 上 4.8%→42.4%
- Wang et al., [A Subgoal-driven Framework for Improving Long-Horizon LLM Agents](https://arxiv.org/abs/2603.19685), 2026 — MiRA：里程碑式密集奖励解决 Agent RL 的稀疏延迟奖励；Gemma3-12B 在 WebArena-Lite 上 6.4%→43.0%
- OpenAI, [Advancing RL for agentic systems](https://openai.com/index/advancing-rl-for-agentic-systems/), 2025 — 远程沙箱环境里做全轨迹级 RL 的前沿案例，奖励来自测试与终端成败
